# Parallel Reduction

Learn to sum N numbers in O(log N) parallel steps using tree-based reduction, optimize with sequential addressing and loop unrolling, and handle arrays larger than one block with multi-pass techniques.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-reduction)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: The Reduction Problem

In [ ]:
!pip install numba

from numba import cuda
import numba
import numpy as np
import math
import time

# ------------------------------------------------------------------
# Basic Parallel Reduction: Sum
# ------------------------------------------------------------------

BLOCK_SIZE = 256

@cuda.jit
def reduce_sum_naive(data, partial_sums, n):
    """Basic tree reduction within a block. Each block produces one partial sum."""
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

    # Load from global to shared memory
    if idx < n:
        shared[tid] = data[idx]
    else:
        shared[tid] = 0.0
    cuda.syncthreads()

    # Tree reduction
    stride = 1
    while stride < BLOCK_SIZE:
        if tid % (2 * stride) == 0 and tid + stride < BLOCK_SIZE:
            shared[tid] += shared[tid + stride]
        stride *= 2
        cuda.syncthreads()

    # Thread 0 writes block result
    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

def gpu_sum(data):
    """Complete reduction: handles arrays larger than one block."""
    n = len(data)
    d_data = cuda.to_device(data)

    while n > 1:
        blocks = math.ceil(n / BLOCK_SIZE)
        d_partial = cuda.device_array(blocks, dtype=np.float32)
        reduce_sum_naive[blocks, BLOCK_SIZE](d_data, d_partial, n)
        d_data = d_partial
        n = blocks

    return d_data.copy_to_host()[0]

# ------------------------------------------------------------------
# Test correctness
# ------------------------------------------------------------------
print("=== Parallel Reduction: Correctness Test ===")
for size in [8, 256, 1000, 10000, 100000, 1000000]:
    data = np.random.randn(size).astype(np.float32)
    gpu_result = gpu_sum(data)
    cpu_result = np.sum(data)
    # Note: slight difference due to FP associativity
    rel_err = abs(gpu_result - cpu_result) / abs(cpu_result) if cpu_result != 0 else 0
    status = "PASS" if rel_err < 1e-5 else "FAIL"
    print(f"  N={size:>10,}: GPU={gpu_result:>12.4f}, CPU={cpu_result:>12.4f}, "
          f"rel_err={rel_err:.2e} {status}")

# ------------------------------------------------------------------
# Visualize the tree reduction steps
# ------------------------------------------------------------------
print("\n=== Reduction Tree Visualization (N=8) ===")
data = np.array([3, 1, 4, 1, 5, 9, 2, 6], dtype=np.float32)
print(f"Input: {data}")

shared = data.copy()
step = 0
stride = 1
while stride < len(shared):
    step += 1
    active = []
    for i in range(len(shared)):
        if i % (2 * stride) == 0 and i + stride < len(shared):
            shared[i] += shared[i + stride]
            active.append(i)
    print(f"Step {step} (stride={stride:>2}): active threads={active}, "
          f"shared={[f'{x:.0f}' for x in shared]}")
    stride *= 2

print(f"Result: {shared[0]:.0f} (expected {np.sum(data):.0f})")

# ------------------------------------------------------------------
# Step count comparison
# ------------------------------------------------------------------
print("\n=== Sequential vs Parallel Steps ===")
print(f"{'N':>12} {'Sequential':>12} {'Parallel':>10} {'Speedup':>10}")
print("-" * 48)
for n in [8, 256, 1024, 1_000_000, 100_000_000]:
    seq = n - 1
    par = math.ceil(math.log2(n))
    print(f"{n:>12,} {seq:>12,} {par:>10} {seq/par:>10.0f}x")

### Lesson example: Naive vs Optimized Reduction

In [ ]:
!pip install numba

from numba import cuda
import numba
import numpy as np
import math
import time

BLOCK_SIZE = 256

# ------------------------------------------------------------------
# Version 1: Interleaved addressing (naive)
# ------------------------------------------------------------------
@cuda.jit
def reduce_v1_interleaved(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * cuda.blockDim.x
    shared[tid] = data[idx] if idx < n else 0.0
    cuda.syncthreads()

    stride = 1
    while stride < BLOCK_SIZE:
        if tid % (2 * stride) == 0 and tid + stride < BLOCK_SIZE:
            shared[tid] += shared[tid + stride]
        stride *= 2
        cuda.syncthreads()

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Version 2: Sequential addressing (fix divergence)
# ------------------------------------------------------------------
@cuda.jit
def reduce_v2_sequential(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * cuda.blockDim.x
    shared[tid] = data[idx] if idx < n else 0.0
    cuda.syncthreads()

    stride = BLOCK_SIZE // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        stride //= 2
        cuda.syncthreads()

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Version 3: First add during load (double elements per block)
# ------------------------------------------------------------------
@cuda.jit
def reduce_v3_first_add(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)

    val = 0.0
    if idx < n:
        val = data[idx]
    if idx + cuda.blockDim.x < n:
        val += data[idx + cuda.blockDim.x]
    shared[tid] = val
    cuda.syncthreads()

    stride = BLOCK_SIZE // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        stride //= 2
        cuda.syncthreads()

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Version 4: Unrolled last warp
# ------------------------------------------------------------------
@cuda.jit
def reduce_v4_unrolled(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)

    val = 0.0
    if idx < n:
        val = data[idx]
    if idx + cuda.blockDim.x < n:
        val += data[idx + cuda.blockDim.x]
    shared[tid] = val
    cuda.syncthreads()

    # Unrolled tree reduction
    if tid < 128:
        shared[tid] += shared[tid + 128]
    cuda.syncthreads()
    if tid < 64:
        shared[tid] += shared[tid + 64]
    cuda.syncthreads()

    # Last warp
    if tid < 32:
        shared[tid] += shared[tid + 32]
        cuda.syncthreads()
        if tid < 16:
            shared[tid] += shared[tid + 16]
        cuda.syncthreads()
        if tid < 8:
            shared[tid] += shared[tid + 8]
        cuda.syncthreads()
        if tid < 4:
            shared[tid] += shared[tid + 4]
        cuda.syncthreads()
        if tid < 2:
            shared[tid] += shared[tid + 2]
        cuda.syncthreads()
        if tid < 1:
            shared[tid] += shared[tid + 1]

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Version 5: Warp shuffle for last warp (fastest)
# ------------------------------------------------------------------
@cuda.jit
def reduce_v5_shuffle(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)

    val = 0.0
    if idx < n:
        val = data[idx]
    if idx + cuda.blockDim.x < n:
        val += data[idx + cuda.blockDim.x]
    shared[tid] = val
    cuda.syncthreads()

    # Tree reduction down to one warp
    if tid < 128:
        shared[tid] += shared[tid + 128]
    cuda.syncthreads()
    if tid < 64:
        shared[tid] += shared[tid + 64]
    cuda.syncthreads()

    # Warp shuffle for the final 32 elements (registers only, no shared mem)
    if tid < 32:
        val = shared[tid]
        val += cuda.shfl_down_sync(0xFFFFFFFF, val, 16)
        val += cuda.shfl_down_sync(0xFFFFFFFF, val, 8)
        val += cuda.shfl_down_sync(0xFFFFFFFF, val, 4)
        val += cuda.shfl_down_sync(0xFFFFFFFF, val, 2)
        val += cuda.shfl_down_sync(0xFFFFFFFF, val, 1)
        if tid == 0:
            partial_sums[cuda.blockIdx.x] = val

# ------------------------------------------------------------------
# Benchmark all versions
# ------------------------------------------------------------------
def reduce_full(kernel, data, n, elements_per_block):
    """Multi-pass reduction using the given kernel."""
    d_data = cuda.to_device(data)
    current_n = n
    while current_n > 1:
        blocks = math.ceil(current_n / elements_per_block)
        d_partial = cuda.device_array(max(blocks, 1), dtype=np.float32)
        kernel[blocks, BLOCK_SIZE](d_data, d_partial, current_n)
        d_data = d_partial
        current_n = blocks
    return d_data.copy_to_host()[0]

def bench_reduce(kernel, data, n, elements_per_block, warmup=10, iters=50):
    for _ in range(warmup):
        reduce_full(kernel, data, n, elements_per_block)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        reduce_full(kernel, data, n, elements_per_block)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000

n = 2_000_000
data = np.random.randn(n).astype(np.float32)
expected = np.sum(data)

print(f"=== Reduction Optimization Benchmark (N={n:,}) ===")
print(f"Expected sum: {expected:.4f}\n")

versions = [
    ("V1: Interleaved", reduce_v1_interleaved, BLOCK_SIZE),
    ("V2: Sequential", reduce_v2_sequential, BLOCK_SIZE),
    ("V3: First-add", reduce_v3_first_add, BLOCK_SIZE * 2),
    ("V4: Unrolled", reduce_v4_unrolled, BLOCK_SIZE * 2),
    ("V5: Warp shuffle", reduce_v5_shuffle, BLOCK_SIZE * 2),
]

print(f"{'Version':<24} {'Time (ms)':<12} {'Speedup':<10} {'Result':<14} {'Error'}")
print("-" * 72)

t_baseline = None
for name, kernel, epb in versions:
    result = reduce_full(kernel, data, n, epb)
    t = bench_reduce(kernel, data, n, epb)
    if t_baseline is None:
        t_baseline = t
    err = abs(result - expected)
    print(f"{name:<24} {t:<12.4f} {t_baseline/t:<10.2f}x {result:<14.4f} {err:.2e}")

# NumPy comparison
start = time.perf_counter()
for _ in range(50):
    _ = np.sum(data)
t_np = (time.perf_counter() - start) / 50 * 1000
print(f"{'NumPy (CPU)':<24} {t_np:<12.4f} {t_baseline/t_np:<10.2f}x {np.sum(data):<14.4f}")

### Lesson example: Multi-Block Reduction

In [ ]:
!pip install numba

from numba import cuda
import numba
import numpy as np
import math
import time

BLOCK_SIZE = 256

# ------------------------------------------------------------------
# Multi-pass reduction (optimized kernel)
# ------------------------------------------------------------------
@cuda.jit
def reduce_optimized(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)

    val = 0.0
    if idx < n:
        val = data[idx]
    if idx + cuda.blockDim.x < n:
        val += data[idx + cuda.blockDim.x]
    shared[tid] = val
    cuda.syncthreads()

    if tid < 128:
        shared[tid] += shared[tid + 128]
    cuda.syncthreads()
    if tid < 64:
        shared[tid] += shared[tid + 64]
    cuda.syncthreads()
    if tid < 32:
        shared[tid] += shared[tid + 32]
        cuda.syncthreads()
        if tid < 16:
            shared[tid] += shared[tid + 16]
        cuda.syncthreads()
        if tid < 8:
            shared[tid] += shared[tid + 8]
        cuda.syncthreads()
        if tid < 4:
            shared[tid] += shared[tid + 4]
        cuda.syncthreads()
        if tid < 2:
            shared[tid] += shared[tid + 2]
        cuda.syncthreads()
        if tid < 1:
            shared[tid] += shared[tid + 1]

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Atomic reduction (single kernel)
# ------------------------------------------------------------------
@cuda.jit
def reduce_atomic(data, result, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)

    val = 0.0
    if idx < n:
        val = data[idx]
    if idx + cuda.blockDim.x < n:
        val += data[idx + cuda.blockDim.x]
    shared[tid] = val
    cuda.syncthreads()

    stride = BLOCK_SIZE // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        stride //= 2
        cuda.syncthreads()

    if tid == 0:
        cuda.atomic.add(result, 0, shared[0])

# ------------------------------------------------------------------
# Multi-element per thread reduction
# ------------------------------------------------------------------
@cuda.jit
def reduce_multi_elem(data, partial_sums, n, elems_per_thread):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    base = cuda.blockIdx.x * cuda.blockDim.x * elems_per_thread + tid

    val = 0.0
    for i in range(elems_per_thread):
        idx = base + i * cuda.blockDim.x
        if idx < n:
            val += data[idx]
    shared[tid] = val
    cuda.syncthreads()

    stride = BLOCK_SIZE // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        stride //= 2
        cuda.syncthreads()

    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def multipass_sum(data, n):
    d_data = cuda.to_device(data)
    current_n = n
    epb = BLOCK_SIZE * 2
    while current_n > 1:
        blocks = math.ceil(current_n / epb)
        d_partial = cuda.device_array(max(blocks, 1), dtype=np.float32)
        reduce_optimized[blocks, BLOCK_SIZE](d_data, d_partial, current_n)
        d_data = d_partial
        current_n = blocks
    return d_data.copy_to_host()[0]

def atomic_sum(data, n):
    d_data = cuda.to_device(data)
    d_result = cuda.to_device(np.zeros(1, dtype=np.float32))
    blocks = math.ceil(n / (BLOCK_SIZE * 2))
    reduce_atomic[blocks, BLOCK_SIZE](d_data, d_result, n)
    return d_result.copy_to_host()[0]

def multi_elem_sum(data, n, ept=8):
    d_data = cuda.to_device(data)
    blocks = math.ceil(n / (BLOCK_SIZE * ept))
    d_partial = cuda.device_array(max(blocks, 1), dtype=np.float32)
    reduce_multi_elem[blocks, BLOCK_SIZE](d_data, d_partial, n, ept)
    if blocks > 1:
        return multipass_sum(d_partial.copy_to_host(), blocks)
    return d_partial.copy_to_host()[0]

# ------------------------------------------------------------------
# Benchmark and compare
# ------------------------------------------------------------------
def bench_fn(fn, data, n, warmup=10, iters=50):
    for _ in range(warmup):
        fn(data, n)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        fn(data, n)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000

for n in [100_000, 1_000_000, 10_000_000]:
    data = np.random.randn(n).astype(np.float32)
    expected = np.sum(data)

    print(f"\n=== N = {n:,} ===")
    print(f"{'Method':<30} {'Time (ms)':<12} {'Result':<14} {'Error':<12} {'BW (GB/s)'}")
    print("-" * 80)

    # Multi-pass
    r = multipass_sum(data, n)
    t = bench_fn(multipass_sum, data, n)
    bw = n * 4 / (t / 1000) / 1e9
    print(f"{'Multi-pass (optimized)':<30} {t:<12.4f} {r:<14.4f} {abs(r-expected):<12.2e} {bw:.1f}")

    # Atomic
    r = atomic_sum(data, n)
    t = bench_fn(atomic_sum, data, n)
    bw = n * 4 / (t / 1000) / 1e9
    print(f"{'Atomic (single kernel)':<30} {t:<12.4f} {r:<14.4f} {abs(r-expected):<12.2e} {bw:.1f}")

    # Multi-element
    r = multi_elem_sum(data, n, ept=8)
    t = bench_fn(lambda d, n: multi_elem_sum(d, n, 8), data, n)
    bw = n * 4 / (t / 1000) / 1e9
    print(f"{'Multi-elem (8/thread)':<30} {t:<12.4f} {r:<14.4f} {abs(r-expected):<12.2e} {bw:.1f}")

    # NumPy reference
    start = time.perf_counter()
    for _ in range(50):
        _ = np.sum(data)
    t_np = (time.perf_counter() - start) / 50 * 1000
    print(f"{'NumPy (CPU)':<30} {t_np:<12.4f} {expected:<14.4f}")

---

## Exercise: Optimized Parallel Sum Reduction vs NumPy


In [ ]:
from numba import cuda
import numba
import numpy as np
import math
import time

BLOCK_SIZE = 256

# --- V1: Basic sequential addressing ---
@cuda.jit
def reduce_v1(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: Load data into shared memory
    # TODO: Tree reduction with sequential addressing
    # TODO: Write block result
    pass

# --- V2: Optimized (first-add + unrolled) ---
@cuda.jit
def reduce_v2(data, partial_sums, n):
    shared = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    tid = cuda.threadIdx.x
    idx = tid + cuda.blockIdx.x * (cuda.blockDim.x * 2)
    # TODO: Load TWO elements per thread and add them
    # TODO: Unrolled tree reduction
    # TODO: Write block result
    pass

# --- V3: Atomic version ---
@cuda.jit
def reduce_atomic(data, result, n):
    # TODO: Same as V2 but use cuda.atomic.add for final result
    pass

def multipass_reduce(kernel, data, n, elements_per_block):
    """Complete multi-pass reduction."""
    # TODO: Implement multi-pass loop
    pass

# TODO: Benchmark all versions vs numpy for sizes [10000, 100000, 1000000, 10000000]
# TODO: Print results table
# TODO: Test atomic determinism (run 10 times, check if results vary)

## Your tasks

Implement an optimized parallel sum reduction kernel and benchmark it against numpy.sum() across different array sizes.

**Requirements:**
1. Implement at least two versions of the reduction kernel:
   - V1: Basic sequential addressing
   - V2: Sequential addressing + first-add during load + unrolled last warp
2. Implement a complete multi-pass reduction function that handles arrays of any size
3. Benchmark both GPU versions and numpy.sum() for sizes: 10K, 100K, 1M, 10M
4. Verify correctness against numpy.sum() (allow relative error < 1e-5)
5. Report a table showing: size, time (ms), bandwidth (GB/s), and speedup vs numpy
6. Also implement an atomic-based reduction and compare its result determinism by running 10 times and checking if results vary

**Hints:**
- Block size of 256 is a good choice
- For first-add, each block processes BLOCK_SIZE * 2 elements
- The multi-pass loop continues until current_n <= 1
- Bandwidth = N * 4 bytes / time (reduction reads each element exactly once)
- Use `cuda.synchronize()` before and after timing
- For the determinism test, run the atomic version 10 times and check if all results are identical

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-reduction) for the solution and discussion.